## Run after pseudobulking in R

additional dependencies
```install.packages("msigdbr")```

In [ ]:
suppressPackageStartupMessages({
    library(dplyr)
    library(readr)
    library(stringr)
    library(edgeR)
    library(limma)
    library(ggplot2)
    library(ggrepel)
    library(patchwork)
    library(reshape2)
    library(viridis)
    library(ggpubr)
    library(umap)
    library(Matrix)
    library(GSVA)
    library(msigdbr)
    library(GSEABase)
    library(VennDiagram)
    library(grid)
    library(RColorBrewer)
    library(rstatix)
    library(emmeans)
    library(fgsea)
})


In [ ]:
run_stratified_limma <- function(tag, keep_idx, extra_covs = EXTRA_COVS) {
  pb_sub  <- pb_meta[keep_idx, , drop = FALSE]
  cnt_sub <- sum_counts[, keep_idx, drop = FALSE]
  dge <- edgeR::DGEList(counts = cnt_sub)
  dge <- edgeR::calcNormFactors(dge)

  # build design WITHOUT GRADE (stratified)
  cov_df <- data.frame(Condition = pb_sub$Condition, check.names = FALSE)

  if (length(extra_covs)) {
    for (cc in extra_covs) {
      if (!cc %in% colnames(pb_sub)) stop(paste("Missing covariate:", cc))
      cov_df[[cc]] <- scale(pb_sub[[cc]])
    }
  }

  # baseline: untreated_17w
  cov_df$Condition <- factor(pb_sub$Condition)
  cov_df$Condition <- stats::relevel(cov_df$Condition, ref = "untreated_17w")

  design <- model.matrix(~ Condition + ., data = cov_df)  # '.' pulls EXTRA_COVS if present

  v <- limma::voom(dge, design, plot = FALSE)

  # write per-stratum voom logCPM for downstream manual work
  write.csv(v$E, file.path(od$de, paste0("voom_logCPM_", tag, ".csv")))

  fit <- limma::lmFit(v, design)
  fit <- limma::eBayes(fit)

  # contrasts (vs untreated_17w). Some may be absent in a stratum; skip those safely.
  coef_map <- c(
    untreated12w_vs_17w          = "Conditionuntreated_12w",
    aCD3_17w_vs_untreated17w     = "Conditionmono_aCD3_17w",
    E2GLP1_17w_vs_untreated17w   = "Conditionmono_E2GLP1_17w",
    combo17w_vs_untreated17w     = "Conditioncombo_aCD3_E2GLP1_17w"
  )

  have_coefs <- intersect(unname(coef_map), colnames(fit$coefficients))
  if (length(have_coefs) == 0) {
    warning(paste0("No estimable Condition coefficients in stratum: ", tag))
    return(invisible(NULL))
  }

  # run outputs only for estimable coefs
  for (nm in names(coef_map)) {
    coef_name <- coef_map[[nm]]
    if (!coef_name %in% have_coefs) {
      message("Skipping ", nm, " (missing in ", tag, ")")
      next
    }
    tt <- limma::topTable(fit, coef = coef_name, number = Inf, sort.by = "P", adjust.method = "BH")
    write.csv(tt, file.path(od$de, paste0("limma_", nm, "_", tag, ".csv")))
  }

  invisible(list(voom = v, fit = fit))
}

make_volcano <- function(tt, title, file){
  dfv <- tt; dfv$gene <- rownames(dfv)
  dfv$signif <- (dfv$adj.P.Val < 0.05 & abs(dfv$logFC) > VOLC_LFC)

  if (length(gene_lists)) {
    cat_map <- setNames(rep(NA_character_, nrow(dfv)), dfv$gene)
    for (nm in names(gene_lists)) {
      gs <- intersect(gene_lists[[nm]], dfv$gene)
      cat_map[gs] <- ifelse(is.na(cat_map[gs]), nm, "Multiple")
    }
    dfv$Category <- factor(cat_map, levels = unique(c(names(gene_lists),"Multiple")))
    pal <- setNames(brewer.pal(max(3, length(levels(dfv$Category))), "Set1")[seq_along(levels(dfv$Category))],
                    levels(dfv$Category))
    p <- ggplot(dfv, aes(logFC, -log10(P.Value))) +
      geom_point(aes(color = Category, shape = signif), alpha = 0.8, size = 1.8, na.rm = TRUE) +
      scale_color_manual(values = pal, na.value = "grey70") +
      scale_shape_manual(values = c(`FALSE`=16, `TRUE`=17)) +
      geom_text_repel(data = subset(dfv, !is.na(Category)),
                      aes(label = gene, color = Category), size = 2.3, max.overlaps = 25) +
      geom_hline(yintercept = -log10(VOLC_P), linetype = 2, color = "grey60") +
      geom_vline(xintercept = c(-VOLC_LFC, VOLC_LFC), linetype = 2, color = "grey60") +
      theme_bw() + labs(title = title, x="log2FC", y="-log10(p)")
  } else {
    p <- ggplot(dfv, aes(logFC, -log10(P.Value))) +
      geom_point(aes(color = signif), alpha = 0.7, size = 1.5) +
      scale_color_manual(values = c(`FALSE`="grey70", `TRUE`="#d62728"), guide = "none") +
      geom_text_repel(data = subset(dfv, signif), aes(label = gene),
                      size = 2.3, max.overlaps = 25) +
      geom_hline(yintercept = -log10(VOLC_P), linetype = 2, color = "grey60") +
      geom_vline(xintercept = c(-VOLC_LFC, VOLC_LFC), linetype = 2, color = "grey60") +
      theme_bw() + labs(title = title, x="log2FC", y="-log10(p)")
  }
  ggsave(file.path(od$de, file), p, width = 6, height = 4.5)
}

make_scatter <- function(df, x, y, shape_var, title) {
  p <- ggplot(df, aes_string(x, y, color = "Donor", shape = shape_var)) +
    geom_point(size = 3, alpha = 0.95) +
    scale_color_discrete(name = "Donor") +
    theme_bw() +
    labs(title = title, x = x, y = y)
  if (include_labels) {
    p <- p + ggrepel::geom_text_repel(aes(label = Pseudobulk),
                                      size = 2.3, max.overlaps = 30)
  }
  p
}

In [ ]:
counts_csv  <- "results/intermediate/pseudobulk_merged_R/pseudobulk_counts_all.csv"
out_root    <- "results/downstream_analysis_R"

filter_regex <- "^beta_pos_immune_neg"   # includes GLP1R variants
min_spots   <- 3
min_counts  <- 10000

# DE volcano thresholds
VOLC_LFC    <- 1
VOLC_P      <- 0.05

EXTRA_COVS  <- c() # for limma

reactome_gmt_path <- 'objects/m2.cp.reactome.v2025.1.Mm.symbols.gmt'
gene_lists <- list()

In [ ]:
counts_out <- read.csv(counts_csv, check.names = FALSE)

meta_cols <- c("sample_id","Condition","laure_region_id","filter","n_spots","total_counts",
               "pct_beta","pct_panleuko","pct_Tcell","pct_CD8pos","pct_cytoCD8","pct_GLP1Rpos")
stopifnot(all(meta_cols %in% colnames(counts_out)))

pb_meta    <- counts_out[, meta_cols, drop = FALSE]
gene_counts<- counts_out[, setdiff(colnames(counts_out), meta_cols), drop = FALSE]

sum_counts <- t(as.matrix(gene_counts))
col_key    <- paste(pb_meta$sample_id, pb_meta$laure_region_id, pb_meta$filter, sep="|")
colnames(sum_counts) <- col_key

pb_meta$GRADE <- stringr::str_match(pb_meta$laure_region_id, ".*_Laure_([^_]+)_.*")[,2]
keep_grade <- !is.na(pb_meta$GRADE) & pb_meta$GRADE != "Q"
pb_meta    <- pb_meta[keep_grade, , drop=FALSE]
sum_counts <- sum_counts[, keep_grade, drop=FALSE]
col_key    <- col_key[keep_grade]
pb_meta$GRADE <- factor(pb_meta$GRADE, levels = c("0","1","2","3"), ordered = TRUE)

keep_qc <- (pb_meta$n_spots >= min_spots) & (pb_meta$total_counts >= min_counts)
pb_meta    <- pb_meta[keep_qc, , drop=FALSE]
sum_counts <- sum_counts[, keep_qc, drop=FALSE]
col_key    <- col_key[keep_qc]

keep_sel <- grepl(filter_regex, pb_meta$filter)
pb_meta    <- pb_meta[keep_sel, , drop=FALSE]
sum_counts <- sum_counts[, keep_sel, drop=FALSE]
col_key    <- col_key[keep_sel]

stopifnot(ncol(sum_counts) == nrow(pb_meta))

pb_meta$Condition <- factor(pb_meta$Condition,
  levels = c("untreated_12w","untreated_17w","mono_aCD3_17w","mono_E2GLP1_17w","combo_aCD3_E2GLP1_17w"))
pb_meta$GRADE <- droplevels(pb_meta$GRADE)

In [ ]:
tag <- gsub("[^A-Za-z0-9_]+","", filter_regex)
od <- list(
  de    = file.path(out_root, paste0("01_DE_", tag)),
  venn  = file.path(out_root, paste0("02_Venn_", tag)),
  dr    = file.path(out_root, paste0("03_DR_", tag)),
  ss    = file.path(out_root, paste0("04_ssGSEA_", tag)),
  hm    = file.path(out_root, paste0("05_Heatmaps_", tag)),
  anova = file.path(out_root, paste0("06_ANOVA_", tag))
)
lapply(od, dir.create, showWarnings = FALSE, recursive = TRUE)

In [ ]:
dge <- DGEList(counts = sum_counts)
dge <- calcNormFactors(dge)

cov_df <- data.frame(Condition = pb_meta$Condition, GRADE = pb_meta$GRADE)

if (length(EXTRA_COVS)) {
  for (cc in EXTRA_COVS) {
    if (!cc %in% colnames(pb_meta)) stop(paste("Missing covariate:", cc))
    cov_df[[cc]] <- scale(pb_meta[[cc]])
  }
}

# make untreated_17w the baseline for contrasts
cov_df$Condition <- factor(pb_meta$Condition)
cov_df$Condition <- stats::relevel(cov_df$Condition, ref = "untreated_17w")

design <- model.matrix(~ Condition + GRADE + ., data = cov_df)  # '.' pulls EXTRA_COVS if any
v <- voom(dge, design, plot = FALSE)
write.csv(v$E, file.path(od$de, "voom_logCPM.csv"))

fit <- lmFit(v, design); fit <- eBayes(fit)

# coefficients for contrasts (Condition terms vs baseline untreated_17w)
coef_map <- c(
  untreated12w_vs_17w          = "Conditionuntreated_12w",
  aCD3_17w_vs_untreated17w     = "Conditionmono_aCD3_17w",
  E2GLP1_17w_vs_untreated17w   = "Conditionmono_E2GLP1_17w",
  combo17w_vs_untreated17w     = "Conditioncombo_aCD3_E2GLP1_17w"
)


In [ ]:
# Define strata
# Treat GRADE as character to be robust to factor/numeric
grade_chr <- as.character(pb_meta$GRADE)

idx_grade0   <- !is.na(grade_chr) & grade_chr == "0"
idx_gradepos <- !is.na(grade_chr) & grade_chr != "0" & grade_chr != "Q"

# Run both strata
run_stratified_limma(tag = "GRADE0_only",     keep_idx = idx_grade0)
run_stratified_limma(tag = "GRADEpos_noQ",    keep_idx = idx_gradepos)

de_sets <- list()  # store up/down sets for venns

for (nm in names(coef_map)) {
  coef_name <- coef_map[[nm]]
  tt <- topTable(fit, coef = coef_name, number = Inf, sort.by = "P", adjust.method = "BH")
  write.csv(tt, file.path(od$de, paste0("limma_", nm, ".csv")))
  make_volcano(tt, paste("Volcano:", nm), paste0("Volcano_", nm, ".pdf"))
  de_sets[[nm]] <- list(
    up   = rownames(tt)[tt$adj.P.Val < 0.05 & tt$logFC >  VOLC_LFC],
    down = rownames(tt)[tt$adj.P.Val < 0.05 & tt$logFC < -VOLC_LFC]
  )
}

In [ ]:
## PCA & UMAP (color = Donor; shape = Condition or GRADE)
include_labels <- FALSE

pca <- prcomp(t(v$E), scale. = TRUE)
pca_df <- data.frame(
  PC1 = pca$x[, 1],
  PC2 = pca$x[, 2],
  Pseudobulk = colnames(v$E),
  Donor = pb_meta$sample_id,
  Condition = pb_meta$Condition,
  GRADE = pb_meta$GRADE
)

set.seed(123)
um <- umap(t(v$E))
um_df <- data.frame(
  UMAP1 = um$layout[, 1],
  UMAP2 = um$layout[, 2],
  Pseudobulk = colnames(v$E),
  Donor = pb_meta$sample_id,
  Condition = pb_meta$Condition,
  GRADE = pb_meta$GRADE
)

# PCA by Condition
p_pca_cond  <- make_scatter(pca_df, "PC1", "PC2", "Condition",
                            "PCA (color = Donor, shape = Condition)")
ggsave(file.path(od$dr, "PCA_byDonor_shapeCondition.pdf"), p_pca_cond, width = 8, height = 5)

# PCA by GRADE
p_pca_grade <- make_scatter(pca_df, "PC1", "PC2", "GRADE",
                            "PCA (color = Donor, shape = GRADE)")
ggsave(file.path(od$dr, "PCA_byDonor_shapeGRADE.pdf"), p_pca_grade, width = 8, height = 5)

# UMAP by Condition
p_um_cond   <- make_scatter(um_df, "UMAP1", "UMAP2", "Condition",
                            "UMAP (color = Donor, shape = Condition)")
ggsave(file.path(od$dr, "UMAP_byDonor_shapeCondition.pdf"), p_um_cond, width = 8, height = 5)

# UMAP by GRADE
p_um_grade  <- make_scatter(um_df, "UMAP1", "UMAP2", "GRADE",
                            "UMAP (color = Donor, shape = GRADE)")
ggsave(file.path(od$dr, "UMAP_byDonor_shapeGRADE.pdf"), p_um_grade, width = 8, height = 5)


In [ ]:
## -------- 6) ssGSEA (Reactome): read GMT if provided, else msigdbr --------
get_reactome_list <- function(){
  if (!is.null(reactome_gmt_path) && file.exists(reactome_gmt_path)) {
    gs <- getGmt(reactome_gmt_path)
    sets <- lapply(gs, geneIds); names(sets) <- sapply(gs, setName); return(sets)
  } else {
    db <- msigdbr(species="Mus musculus", category="C2", subcategory="CP:REACTOME")
    split(db$gene_symbol, db$gs_name)
  }
}
reactome_list <- get_reactome_list()

# ssGSEA on genes x samples; we can use voom logCPM as input (continuous)
ss <- gsva(v$E, reactome_list, method="ssgsea", verbose=FALSE)
scores_df <- as.data.frame(t(ss))  # samples x pathways
scores_df$Sample    <- rownames(scores_df)
scores_df$Condition <- pb_meta$Condition
scores_df$GRADE     <- pb_meta$GRADE
write.csv(scores_df, file.path(od$ss, "ssGSEA_Reactome_scores.csv"), row.names=FALSE)

In [ ]:
genelists_manual <- list(
  SachsB1ike = c("Ins1","Ins2","Ucn3","Trpm5","Slc2a2","Slc30a8","G6pc2","Sytl4","Nkx6-1","Neurod1","Pdx1","Nkx2-2","Pax6","Ils1"),
  SachsmSTZshort  = c("Pcsk1","Pam","Cpe","Gast","Iapp","Mafb","Chgb","Rbp4","Adlh1a3"),
  SachsmSTZext = c("Pcsk1","Pam","Cpe","Gast","Iapp","Mafb","Chgb","Rbp4","Adlh1a3","Slc5a10","Phlda3","Sorcs2","Dpp6","Ldlrad3","Nrsn1","Bcam","Tmem212","Clmp","Sh2d5",
                   "Slc39a11","Ptger3","Ache","Tenm4","Aldh1a3","Tagln3","Pabpc1l","Aldob","Gsto2","Ttc25","Gast","Cck","Chtrc1","Cartpt","Smoc1","Prss23","Gm2115","Pcp4l1","RP21-477O13.1")
)

reactome_gmt <- reactome_gmt_path
reactome_sets <- fgsea::gmtPathways(reactome_gmt)

expr <- v$E
genes_in_expr <- rownames(expr)

all_sets <- c(genelists_manual, reactome_sets)

# Keep sets with at least 2 genes present in expr (GSVA/ssGSEA needs >1)
all_sets <- lapply(all_sets, function(gs) intersect(gs, genes_in_expr))
len_ok <- vapply(all_sets, length, integer(1)) >= 2
all_sets <- all_sets[len_ok]
if (length(all_sets) == 0) stop("No Reactome/manual sets have ≥2 genes present in expr.")

ssgsea_mat <- gsva(as.matrix(expr), all_sets, method = "ssgsea", verbose = FALSE)

scores_df <- data.frame(Sample = colnames(expr), check.names = FALSE)
scores_df <- cbind(scores_df, t(ssgsea_mat)[scores_df$Sample, , drop = FALSE])
scores_df$Condition <- pb_meta$Condition
scores_df$GRADE     <- pb_meta$GRADE

out_dir <- file.path(od$anova, "tables")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
out_csv <- file.path(out_dir, "ssgsea_scores_manual_plus_reactome_wide.csv")
write.csv(scores_df, out_csv, row.names = FALSE)
message("✓ wrote: ", out_csv)


## At this point, you can proceed to Python_downstream.ipynb to generate Figures